In [0]:
# Step 1: Read Store data from Bronze

store_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(
        "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/store/"
    )

display(store_df)

In [0]:
# Step 2: Convert Store column headers to snake_case
import re

for column in store_df.columns:
    store_df = store_df.withColumnRenamed(
        column,
        to_snake_case(column)
    )

print(store_df.columns)

In [0]:
# Step 2: Convert Store column headers to snake_case

for column in store_df.columns:
    store_df = store_df.withColumnRenamed(
        column,
        to_snake_case(column)
    )

print(store_df.columns)

In [0]:
# Step 3A: Check Store columns

print(store_df.columns)

In [0]:
# Step 3: Correct store_id column name

store_df = store_df.withColumnRenamed(
    "store__i_d",
    "store_id"
)

print(store_df.columns)

In [0]:
# Step 4: Create store_category from email_address

from pyspark.sql.functions import col, split

store_df = store_df.withColumn(
    "store_category",
    split(
        split(col("email_address"), "@").getItem(1),
        "\\."
    ).getItem(0)
)

display(store_df)

In [0]:
# Step 5A: Reload Store data from Bronze

store_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(
        "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/store/"
    )

In [0]:
# Step 5B: Convert columns to snake_case

for column in store_df.columns:
    store_df = store_df.withColumnRenamed(
        column,
        to_snake_case(column)
    )

store_df = store_df.withColumnRenamed(
    "store__i_d",
    "store_id"
)

In [0]:

for column in store_df.columns:
    store_df = store_df.withColumnRenamed(
        column,
        to_snake_case(column)
    )

store_df = store_df.withColumnRenamed(
    "store__i_d",
    "store_id"
)

In [0]:
# Step 5C: Create store_category

from pyspark.sql.functions import col, split

store_df = store_df.withColumn(
    "store_category",
    split(
        split(col("email_address"), "@").getItem(1),
        "\\."
    ).getItem(0)
)

In [0]:
# Step 5D: Format created_at and updated_at to yyyy-MM-dd

from pyspark.sql.functions import to_date, col

store_df = store_df.withColumn(
    "created_at",
    to_date(col("created_at"), "dd-MM-yyyy")
)

store_df = store_df.withColumn(
    "updated_at",
    to_date(col("updated_at"), "dd-MM-yyyy")
)

display(store_df)

In [0]:
# Step 6: Import DeltaTable

from delta.tables import DeltaTable

In [0]:
# Step 7: Upsert Store data to Silver

silver_path = "abfss://silver@adfassignment7.dfs.core.windows.net/sales_view/store/"

if DeltaTable.isDeltaTable(spark, silver_path):

    target = DeltaTable.forPath(
        spark,
        silver_path
    )

    target.alias("target") \
        .merge(
            store_df.alias("source"),
            "target.store_id = source.store_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    print("Store data upserted successfully to Silver")

else:

    store_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(silver_path)

    print("Store Silver Delta table created successfully")